# 602 — SHAP Attribution and Stability Analysis

## Objective

Characterize program-level attribution, resampling stability, and lineage consistency for the frozen notebook-601 GDSC predictive models that satisfied the prospectively defined internal predictive-validity gate.

This notebook treats SHAP values as explanations of fitted-model behavior. It does not establish causal biological effects, drug-response mechanisms, therapeutic efficacy, validated biomarkers, validated therapeutic targets, clinical predictiveness, or external cross-screen reproducibility.

## Analytical boundary

Notebook 602 is restricted to the 125 GDSC drugs with `shap_eligible == True` in the frozen notebook-601 drug-level handoff.

The notebook consumes the persisted notebook-601 out-of-fold predictions and fitted-state handoffs. It does not:

* refit notebook-601 predictive models;
* regenerate notebook-601 cross-validation partitions;
* recompute predictive eligibility;
* introduce alternative model families;
* add gene-level features or program × lineage interactions; or
* perform a separate SHAP analysis of the notebook-601 LOLO models.

Primary attribution is calculated only for the persisted held-out predictions from the frozen `5-fold lineage-stratified cross-validation × 5 repeats` design.

The prospectively frozen notebook-602 specification was established on `2026-09-21`, before inspection of any notebook-602 SHAP attribution result.

## Attribution framework

Notebook 602 explains the frozen lineage-plus-program model:

`LN_IC50 ~ C(OncotreeLineage) + CONSENSUS_TX_01 + CONSENSUS_TX_02 + CONSENSUS_TX_03`

using exact interventional linear SHAP derived from the persisted fold-specific model state.

Primary biological attribution remains at consensus-program resolution:

* `CONSENSUS_TX_01`;
* `CONSENSUS_TX_02`; and
* `CONSENSUS_TX_03`.

Lineage is retained as a grouped model-attribution component for contextual comparison.

Primary program-attribution magnitude is summarized as `mean(abs(SHAP))` across the complete out-of-fold sample within each repeat and then by the median across the five repeats.

Attribution stability is characterized continuously rather than through a new binary gate. Lineage consistency is evaluated descriptively by comparing pooled and lineage-balanced attribution summaries without fitting lineage-specific models or interactions.

## Biological contextualization

Attributed programs may subsequently be linked to frozen Phase 4 and Phase 4B gene weights, pathway annotations, epigenetic-regulator enrichment, tumor-side methylation context, and secondary genomic or locus-level methylation-expression context.

This mapping is biological contextualization of program-level attribution.

It must not be represented as gene-level SHAP evidence or used to redistribute program-level SHAP values across constituent genes.

## Evidence isolation

* **GDSC:** developmental/internal resource supplying the frozen notebook-601 predictive models and notebook-602 attribution objects.
* **CTRP and PRISM:** remain sealed for pharmacogenomic outcomes during notebook 602 and are reserved for notebook 603 cross-screen replication.
* **Notebook 600 associations:** do not determine notebook-602 attribution eligibility, aggregation, stability, or lineage-consistency rules.
* **Phase 5:** functional-vulnerability evidence is not used to select drugs or programs or to rescue weak attribution.

Residual proliferation and other unresolved cell-line confounding remain explicit limitations inherited from the frozen predictive analysis rather than grounds for post hoc covariate construction.

SHAP attribution, coefficient direction, attribution stability, lineage consistency, and biological contextualization remain distinct evidence dimensions and must not be interpreted as causal biological mechanism.


In [1]:
# =============================================================================
# Imports
# =============================================================================

import json

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Resolve frozen notebook-601 handoffs
# =============================================================================

artifact_registry = load_artifact_registry()

INPUT_ARTIFACT_IDS = (
    "phase6.601.primary_oof_predictions",
    "phase6.601.primary_program_fold_parameters",
    "phase6.601.primary_program_fold_lineage_effects",
    "phase6.601.drug_level_results",
    "phase6.601.primary_cv_partitions",
    "phase6.601.primary_repeat_performance",
    "phase6.601.analysis_metadata",
)

input_paths = {
    artifact_id: resolve_artifact_path(
        artifact_registry,
        artifact_id,
    )
    for artifact_id in INPUT_ARTIFACT_IDS
}

for artifact_id, path in input_paths.items():
    print(
        f"{artifact_id}: "
        f"{project_relative_path(path)}"
    )

phase6.601.primary_oof_predictions: data/processed/pharmacogenomic_contexts/601_primary_oof_predictions.parquet
phase6.601.primary_program_fold_parameters: data/processed/pharmacogenomic_contexts/601_primary_program_fold_parameters.parquet
phase6.601.primary_program_fold_lineage_effects: data/processed/pharmacogenomic_contexts/601_primary_program_fold_lineage_effects.parquet
phase6.601.drug_level_results: data/processed/pharmacogenomic_contexts/601_drug_level_results.csv
phase6.601.primary_cv_partitions: data/processed/pharmacogenomic_contexts/601_primary_cv_partitions.parquet
phase6.601.primary_repeat_performance: data/processed/pharmacogenomic_contexts/601_primary_repeat_performance.csv
phase6.601.analysis_metadata: data/processed/pharmacogenomic_contexts/601_analysis_metadata.json


In [3]:
# =============================================================================
# Load frozen notebook-601 attribution inputs
# =============================================================================

primary_oof_predictions = pd.read_parquet(
    input_paths[
        "phase6.601.primary_oof_predictions"
    ]
)

primary_program_fold_parameters = pd.read_parquet(
    input_paths[
        "phase6.601.primary_program_fold_parameters"
    ]
)

primary_program_fold_lineage_effects = pd.read_parquet(
    input_paths[
        "phase6.601.primary_program_fold_lineage_effects"
    ]
)

drug_level_results = pd.read_csv(
    input_paths[
        "phase6.601.drug_level_results"
    ]
)

print(
    "Primary OOF predictions shape:",
    primary_oof_predictions.shape,
)
print(
    "Primary fold parameters shape:",
    primary_program_fold_parameters.shape,
)
print(
    "Primary fold lineage effects shape:",
    primary_program_fold_lineage_effects.shape,
)
print(
    "Drug-level results shape:",
    drug_level_results.shape,
)

Primary OOF predictions shape: (680880, 11)
Primary fold parameters shape: (7025, 15)
Primary fold lineage effects shape: (74025, 8)
Drug-level results shape: (281, 24)


In [4]:
# =============================================================================
# Restrict analysis to the frozen notebook-602 attribution cohort
# =============================================================================

shap_eligible_drugs = (
    drug_level_results.loc[
        drug_level_results["shap_eligible"].astype(bool),
        "DRUG_ID",
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

eligible_drug_ids = set(shap_eligible_drugs)

oof_attribution_input = primary_oof_predictions.loc[
    primary_oof_predictions["DRUG_ID"].isin(
        eligible_drug_ids
    )
].copy()

fold_parameter_input = (
    primary_program_fold_parameters.loc[
        primary_program_fold_parameters["DRUG_ID"].isin(
            eligible_drug_ids
        )
    ].copy()
)

fold_lineage_effect_input = (
    primary_program_fold_lineage_effects.loc[
        primary_program_fold_lineage_effects["DRUG_ID"].isin(
            eligible_drug_ids
        )
    ].copy()
)

print(
    "Frozen SHAP-eligible drugs:",
    len(shap_eligible_drugs),
)
print(
    "OOF attribution input shape:",
    oof_attribution_input.shape,
)
print(
    "Fold-parameter input shape:",
    fold_parameter_input.shape,
)
print(
    "Fold-lineage-effect input shape:",
    fold_lineage_effect_input.shape,
)

print(
    "\nDrugs represented in OOF attribution input:",
    oof_attribution_input["DRUG_ID"].nunique(),
)
print(
    "Drugs represented in fold parameters:",
    fold_parameter_input["DRUG_ID"].nunique(),
)
print(
    "Drugs represented in fold lineage effects:",
    fold_lineage_effect_input["DRUG_ID"].nunique(),
)

Frozen SHAP-eligible drugs: 125
OOF attribution input shape: (298195, 11)
Fold-parameter input shape: (3125, 15)
Fold-lineage-effect input shape: (32650, 8)

Drugs represented in OOF attribution input: 125
Drugs represented in fold parameters: 125
Drugs represented in fold lineage effects: 125


In [5]:
# =============================================================================
# Validate frozen attribution-cohort alignment
# =============================================================================

PROGRAM_COLUMNS = [
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
]

OOF_KEY_COLUMNS = [
    "DRUG_ID",
    "ModelID",
    "repeat",
]

FOLD_KEY_COLUMNS = [
    "DRUG_ID",
    "repeat",
    "fold",
]

LINEAGE_EFFECT_KEY_COLUMNS = [
    "DRUG_ID",
    "repeat",
    "fold",
    "OncotreeLineage",
]

oof_drug_ids = set(
    oof_attribution_input["DRUG_ID"].unique()
)
parameter_drug_ids = set(
    fold_parameter_input["DRUG_ID"].unique()
)
lineage_effect_drug_ids = set(
    fold_lineage_effect_input["DRUG_ID"].unique()
)

drug_sets_aligned = (
    eligible_drug_ids
    == oof_drug_ids
    == parameter_drug_ids
    == lineage_effect_drug_ids
)

expected_fold_parameter_rows = (
    len(shap_eligible_drugs) * 5 * 5
)

print(
    "Attribution drug sets aligned:",
    drug_sets_aligned,
)
print(
    "One OOF row per drug × ModelID × repeat:",
    not oof_attribution_input.duplicated(
        OOF_KEY_COLUMNS
    ).any(),
)
print(
    "One parameter row per drug × repeat × fold:",
    not fold_parameter_input.duplicated(
        FOLD_KEY_COLUMNS
    ).any(),
)
print(
    "One lineage-effect row per drug × repeat × fold × lineage:",
    not fold_lineage_effect_input.duplicated(
        LINEAGE_EFFECT_KEY_COLUMNS
    ).any(),
)
print(
    "Fold-parameter rows:",
    len(fold_parameter_input),
    "/ expected:",
    expected_fold_parameter_rows,
)
print(
    "Missing OOF program scores:",
    int(
        oof_attribution_input[
            PROGRAM_COLUMNS
        ]
        .isna()
        .any(axis=1)
        .sum()
    ),
)

Attribution drug sets aligned: True
One OOF row per drug × ModelID × repeat: True
One parameter row per drug × repeat × fold: True
One lineage-effect row per drug × repeat × fold × lineage: True
Fold-parameter rows: 3125 / expected: 3125
Missing OOF program scores: 0


In [6]:
# =============================================================================
# Attach persisted fold-specific model parameters
# =============================================================================

attribution_rows = oof_attribution_input.merge(
    fold_parameter_input,
    on=[
        "DRUG_ID",
        "repeat",
        "fold",
    ],
    how="left",
    validate="many_to_one",
)

print(
    "Attribution rows after fold-parameter merge:",
    attribution_rows.shape,
)

print(
    "Rows missing persisted fold parameters:",
    attribution_rows[
        "program_intercept"
    ].isna().sum(),
)

Attribution rows after fold-parameter merge: (298195, 23)
Rows missing persisted fold parameters: 0


In [7]:
# =============================================================================
# Attach persisted fold-specific lineage effects
# =============================================================================

attribution_rows = attribution_rows.merge(
    fold_lineage_effect_input[
        [
            "DRUG_ID",
            "repeat",
            "fold",
            "OncotreeLineage",
            "lineage_effect",
        ]
    ],
    on=[
        "DRUG_ID",
        "repeat",
        "fold",
        "OncotreeLineage",
    ],
    how="left",
    validate="many_to_one",
)

print(
    "Attribution rows after lineage-effect merge:",
    attribution_rows.shape,
)

print(
    "Rows missing persisted lineage effects:",
    attribution_rows["lineage_effect"].isna().sum(),
)

Attribution rows after lineage-effect merge: (298195, 24)
Rows missing persisted lineage effects: 0


In [8]:
# =============================================================================
# Calculate exact held-out linear SHAP attributions
# =============================================================================

attribution_rows["expected_value"] = (
    attribution_rows["program_intercept"]
    + attribution_rows["train_mean_lineage_effect"]
)

attribution_rows["phi_lineage"] = (
    attribution_rows["lineage_effect"]
    - attribution_rows["train_mean_lineage_effect"]
)

for program in PROGRAM_COLUMNS:
    beta_column = f"beta_{program}"
    mean_column = f"train_mean_{program}"
    phi_column = f"phi_{program}"

    attribution_rows["expected_value"] += (
        attribution_rows[beta_column]
        * attribution_rows[mean_column]
    )

    attribution_rows[phi_column] = (
        attribution_rows[beta_column]
        * (
            attribution_rows[program]
            - attribution_rows[mean_column]
        )
    )

print(
    "Held-out attribution rows:",
    len(attribution_rows),
)
print(
    "Program attribution columns:",
    [
        f"phi_{program}"
        for program in PROGRAM_COLUMNS
    ],
)

Held-out attribution rows: 298195
Program attribution columns: ['phi_CONSENSUS_TX_01', 'phi_CONSENSUS_TX_02', 'phi_CONSENSUS_TX_03']


In [9]:
# =============================================================================
# Validate exact additive reconstruction of persisted predictions
# =============================================================================

phi_program_columns = [
    f"phi_{program}"
    for program in PROGRAM_COLUMNS
]

attribution_rows["reconstructed_prediction"] = (
    attribution_rows["expected_value"]
    + attribution_rows["phi_lineage"]
    + attribution_rows[phi_program_columns].sum(axis=1)
)

prediction_error = (
    attribution_rows["reconstructed_prediction"]
    - attribution_rows["prediction_program"]
)

reconstruction_matches = np.allclose(
    attribution_rows["reconstructed_prediction"],
    attribution_rows["prediction_program"],
    rtol=1e-12,
    atol=1e-12,
)

print(
    "Exact additive reconstruction:",
    reconstruction_matches,
)
print(
    "Maximum absolute reconstruction error:",
    np.abs(prediction_error).max(),
)

if not reconstruction_matches:
    raise RuntimeError(
        "Notebook-602 attribution decomposition does not "
        "reconstruct persisted notebook-601 predictions."
    )

Exact additive reconstruction: True
Maximum absolute reconstruction error: 2.6645352591003757e-15


In [10]:
# =============================================================================
# Summarize program attribution within each OOF repeat
# =============================================================================

program_attribution_long = attribution_rows.melt(
    id_vars=[
        "DRUG_ID",
        "ModelID",
        "OncotreeLineage",
        "repeat",
        "fold",
    ],
    value_vars=phi_program_columns,
    var_name="program",
    value_name="shap_value",
)

program_attribution_long["program"] = (
    program_attribution_long["program"]
    .str.removeprefix("phi_")
)

program_attribution_long["abs_shap"] = (
    program_attribution_long["shap_value"].abs()
)

repeat_program_attribution = (
    program_attribution_long
    .groupby(
        [
            "DRUG_ID",
            "repeat",
            "program",
        ],
        as_index=False,
    )
    .agg(
        n_models=("ModelID", "size"),
        mean_abs_shap=("abs_shap", "mean"),
        median_abs_shap=("abs_shap", "median"),
    )
)

print(
    "Long OOF program-attribution rows:",
    program_attribution_long.shape,
)
print(
    "Repeat-level program-attribution rows:",
    repeat_program_attribution.shape,
)

repeat_program_attribution.head()

Long OOF program-attribution rows: (894585, 8)
Repeat-level program-attribution rows: (1875, 6)


,DRUG_ID,repeat,program,n_models,mean_abs_shap,median_abs_shap
0,1003,1,CONSENSUS_TX_01,556,0.495195,0.361675
1,1003,1,CONSENSUS_TX_02,556,0.280192,0.250155
2,1003,1,CONSENSUS_TX_03,556,0.098533,0.074176
3,1003,2,CONSENSUS_TX_01,556,0.493774,0.354222
4,1003,2,CONSENSUS_TX_02,556,0.280817,0.260014


In [11]:
# =============================================================================
# Identify the dominant attributed program within each repeat
# =============================================================================

repeat_program_attribution["is_dominant_program"] = (
    repeat_program_attribution["mean_abs_shap"]
    == repeat_program_attribution.groupby(
        [
            "DRUG_ID",
            "repeat",
        ]
    )["mean_abs_shap"].transform("max")
)

dominant_program_by_repeat = (
    repeat_program_attribution.loc[
        repeat_program_attribution[
            "is_dominant_program"
        ]
    ]
    [
        [
            "DRUG_ID",
            "repeat",
            "program",
            "mean_abs_shap",
        ]
    ]
    .rename(
        columns={
            "program": "dominant_program",
            "mean_abs_shap": "dominant_mean_abs_shap",
        }
    )
    .reset_index(drop=True)
)

print(
    "Drug × repeat dominant-program rows:",
    dominant_program_by_repeat.shape,
)

print(
    "Drug × repeat combinations:",
    repeat_program_attribution[
        ["DRUG_ID", "repeat"]
    ]
    .drop_duplicates()
    .shape[0],
)

dominant_program_by_repeat.head()

Drug × repeat dominant-program rows: (625, 4)
Drug × repeat combinations: 625


,DRUG_ID,repeat,dominant_program,dominant_mean_abs_shap
0,1003,1,CONSENSUS_TX_01,0.495195
1,1003,2,CONSENSUS_TX_01,0.493774
2,1003,3,CONSENSUS_TX_01,0.495094
3,1003,4,CONSENSUS_TX_01,0.493587
4,1003,5,CONSENSUS_TX_01,0.492472


In [12]:
# =============================================================================
# Summarize program attribution magnitude and repeat stability
# =============================================================================

drug_program_attribution_summary = (
    repeat_program_attribution
    .groupby(
        [
            "DRUG_ID",
            "program",
        ],
        as_index=False,
    )
    .agg(
        median_repeat_mean_abs_shap=(
            "mean_abs_shap",
            "median",
        ),
        q25_repeat_mean_abs_shap=(
            "mean_abs_shap",
            lambda x: x.quantile(0.25),
        ),
        q75_repeat_mean_abs_shap=(
            "mean_abs_shap",
            lambda x: x.quantile(0.75),
        ),
        min_repeat_mean_abs_shap=(
            "mean_abs_shap",
            "min",
        ),
        max_repeat_mean_abs_shap=(
            "mean_abs_shap",
            "max",
        ),
        median_repeat_median_abs_shap=(
            "median_abs_shap",
            "median",
        ),
        n_repeats=(
            "repeat",
            "nunique",
        ),
        dominant_repeat_count=(
            "is_dominant_program",
            "sum",
        ),
    )
)

drug_program_attribution_summary[
    "iqr_repeat_mean_abs_shap"
] = (
    drug_program_attribution_summary[
        "q75_repeat_mean_abs_shap"
    ]
    - drug_program_attribution_summary[
        "q25_repeat_mean_abs_shap"
    ]
)

drug_program_attribution_summary[
    "dominant_repeat_fraction"
] = (
    drug_program_attribution_summary[
        "dominant_repeat_count"
    ]
    / drug_program_attribution_summary[
        "n_repeats"
    ]
)

print(
    "Drug × program attribution-summary rows:",
    drug_program_attribution_summary.shape,
)

drug_program_attribution_summary.head()

Drug × program attribution-summary rows: (375, 12)


,DRUG_ID,program,median_repeat_mean_abs_shap,q25_repeat_mean_abs_shap,q75_repeat_mean_abs_shap,min_repeat_mean_abs_shap,max_repeat_mean_abs_shap,median_repeat_median_abs_shap,n_repeats,dominant_repeat_count,iqr_repeat_mean_abs_shap,dominant_repeat_fraction
0,1003,CONSENSUS_TX_01,0.493774,0.493587,0.495094,0.492472,0.495195,0.354222,5,5,0.001507,1.0
1,1003,CONSENSUS_TX_02,0.280817,0.280192,0.282048,0.279926,0.285687,0.250155,5,0,0.001856,0.0
2,1003,CONSENSUS_TX_03,0.098533,0.096758,0.100502,0.095830,0.102058,0.068122,5,0,0.003744,0.0
3,1004,CONSENSUS_TX_01,0.285828,0.283541,0.288258,0.282115,0.288843,0.209169,5,0,0.004717,0.0
4,1004,CONSENSUS_TX_02,0.706181,0.706094,0.707038,0.702981,0.710803,0.650627,5,5,0.000944,1.0


In [13]:
# =============================================================================
# Summarize persisted program coefficients across fitted folds
# =============================================================================

beta_columns = [
    f"beta_{program}"
    for program in PROGRAM_COLUMNS
]

program_coefficients_long = (
    fold_parameter_input
    .melt(
        id_vars=[
            "DRUG_ID",
            "repeat",
            "fold",
        ],
        value_vars=beta_columns,
        var_name="program",
        value_name="coefficient",
    )
)

program_coefficients_long["program"] = (
    program_coefficients_long["program"]
    .str.removeprefix("beta_")
)

coefficient_summary = (
    program_coefficients_long
    .groupby(
        [
            "DRUG_ID",
            "program",
        ],
        as_index=False,
    )
    .agg(
        n_fitted_models=(
            "coefficient",
            "size",
        ),
        median_coefficient=(
            "coefficient",
            "median",
        ),
        q25_coefficient=(
            "coefficient",
            lambda x: x.quantile(0.25),
        ),
        q75_coefficient=(
            "coefficient",
            lambda x: x.quantile(0.75),
        ),
        fraction_positive=(
            "coefficient",
            lambda x: (x > 0).mean(),
        ),
        fraction_negative=(
            "coefficient",
            lambda x: (x < 0).mean(),
        ),
    )
)

coefficient_summary["iqr_coefficient"] = (
    coefficient_summary["q75_coefficient"]
    - coefficient_summary["q25_coefficient"]
)

print(
    "Long fitted-coefficient rows:",
    program_coefficients_long.shape,
)
print(
    "Drug × program coefficient-summary rows:",
    coefficient_summary.shape,
)

coefficient_summary.head()

Long fitted-coefficient rows: (9375, 5)
Drug × program coefficient-summary rows: (375, 9)


,DRUG_ID,program,n_fitted_models,median_coefficient,q25_coefficient,q75_coefficient,fraction_positive,fraction_negative,iqr_coefficient
0,1003,CONSENSUS_TX_01,25,-0.568176,-0.653374,-0.496686,0.00,1.00,0.156688
1,1003,CONSENSUS_TX_02,25,-0.354836,-0.367466,-0.303957,0.00,1.00,0.063509
2,1003,CONSENSUS_TX_03,25,0.164435,0.091670,0.178533,0.96,0.04,0.086863
3,1004,CONSENSUS_TX_01,25,-0.372261,-0.442923,-0.226922,0.00,1.00,0.216001
4,1004,CONSENSUS_TX_02,25,-0.892103,-1.026734,-0.864548,0.00,1.00,0.162186


In [14]:
# =============================================================================
# Combine attribution magnitude and coefficient-stability summaries
# =============================================================================

drug_program_summary = (
    drug_program_attribution_summary
    .merge(
        coefficient_summary,
        on=[
            "DRUG_ID",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
)

print(
    "Integrated drug × program summary shape:",
    drug_program_summary.shape,
)

print(
    "Rows missing coefficient summaries:",
    drug_program_summary[
        "median_coefficient"
    ].isna().sum(),
)

drug_program_summary.head()

Integrated drug × program summary shape: (375, 19)
Rows missing coefficient summaries: 0


,DRUG_ID,program,median_repeat_mean_abs_shap,q25_repeat_mean_abs_shap,q75_repeat_mean_abs_shap,min_repeat_mean_abs_shap,max_repeat_mean_abs_shap,median_repeat_median_abs_shap,n_repeats,dominant_repeat_count,iqr_repeat_mean_abs_shap,dominant_repeat_fraction,n_fitted_models,median_coefficient,q25_coefficient,q75_coefficient,fraction_positive,fraction_negative,iqr_coefficient
0,1003,CONSENSUS_TX_01,0.493774,0.493587,0.495094,0.492472,0.495195,0.354222,5,5,0.001507,1.0,25,-0.568176,-0.653374,-0.496686,0.00,1.00,0.156688
1,1003,CONSENSUS_TX_02,0.280817,0.280192,0.282048,0.279926,0.285687,0.250155,5,0,0.001856,0.0,25,-0.354836,-0.367466,-0.303957,0.00,1.00,0.063509
2,1003,CONSENSUS_TX_03,0.098533,0.096758,0.100502,0.095830,0.102058,0.068122,5,0,0.003744,0.0,25,0.164435,0.091670,0.178533,0.96,0.04,0.086863
3,1004,CONSENSUS_TX_01,0.285828,0.283541,0.288258,0.282115,0.288843,0.209169,5,0,0.004717,0.0,25,-0.372261,-0.442923,-0.226922,0.00,1.00,0.216001
4,1004,CONSENSUS_TX_02,0.706181,0.706094,0.707038,0.702981,0.710803,0.650627,5,5,0.000944,1.0,25,-0.892103,-1.026734,-0.864548,0.00,1.00,0.162186


In [15]:
# =============================================================================
# Summarize program attribution within lineage and repeat
# =============================================================================

lineage_program_attribution = (
    program_attribution_long
    .groupby(
        [
            "DRUG_ID",
            "repeat",
            "OncotreeLineage",
            "program",
        ],
        as_index=False,
    )
    .agg(
        n_models=("ModelID", "size"),
        lineage_mean_abs_shap=(
            "abs_shap",
            "mean",
        ),
    )
)

print(
    "Lineage × repeat × program attribution rows:",
    lineage_program_attribution.shape,
)

print(
    "Represented lineages:",
    lineage_program_attribution[
        "OncotreeLineage"
    ].nunique(),
)

lineage_program_attribution.head()

Lineage × repeat × program attribution rows: (19590, 6)
Represented lineages: 11


,DRUG_ID,repeat,OncotreeLineage,program,n_models,lineage_mean_abs_shap
0,1003,1,Bowel,CONSENSUS_TX_01,43,0.237964
1,1003,1,Bowel,CONSENSUS_TX_02,43,0.264308
2,1003,1,Bowel,CONSENSUS_TX_03,43,0.114773
3,1003,1,Breast,CONSENSUS_TX_01,46,0.323540
4,1003,1,Breast,CONSENSUS_TX_02,46,0.244732


In [16]:
# =============================================================================
# Calculate lineage-specific program attribution
# =============================================================================

lineage_program_attribution = (
    program_attribution_long
    .groupby(
        [
            "DRUG_ID",
            "repeat",
            "OncotreeLineage",
            "program",
        ],
        as_index=False,
    )
    .agg(
        n_models=("ModelID", "size"),
        lineage_mean_abs_shap=(
            "abs_shap",
            "mean",
        ),
    )
)

print(
    "Lineage-level program-attribution rows:",
    lineage_program_attribution.shape,
)

lineage_program_attribution.head()

Lineage-level program-attribution rows: (19590, 6)


,DRUG_ID,repeat,OncotreeLineage,program,n_models,lineage_mean_abs_shap
0,1003,1,Bowel,CONSENSUS_TX_01,43,0.237964
1,1003,1,Bowel,CONSENSUS_TX_02,43,0.264308
2,1003,1,Bowel,CONSENSUS_TX_03,43,0.114773
3,1003,1,Breast,CONSENSUS_TX_01,46,0.323540
4,1003,1,Breast,CONSENSUS_TX_02,46,0.244732


In [17]:
# =============================================================================
# Compare pooled and lineage-balanced program attribution
# =============================================================================

lineage_balanced_attribution = (
    lineage_program_attribution
    .groupby(
        [
            "DRUG_ID",
            "repeat",
            "program",
        ],
        as_index=False,
    )
    .agg(
        n_lineages=(
            "OncotreeLineage",
            "nunique",
        ),
        lineage_balanced_mean_abs_shap=(
            "lineage_mean_abs_shap",
            "mean",
        ),
    )
)

repeat_lineage_consistency = (
    repeat_program_attribution[
        [
            "DRUG_ID",
            "repeat",
            "program",
            "mean_abs_shap",
        ]
    ]
    .rename(
        columns={
            "mean_abs_shap": "pooled_mean_abs_shap",
        }
    )
    .merge(
        lineage_balanced_attribution,
        on=[
            "DRUG_ID",
            "repeat",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
)

repeat_lineage_consistency[
    "pooled_minus_lineage_balanced"
] = (
    repeat_lineage_consistency[
        "pooled_mean_abs_shap"
    ]
    - repeat_lineage_consistency[
        "lineage_balanced_mean_abs_shap"
    ]
)

print(
    "Repeat-level lineage-consistency rows:",
    repeat_lineage_consistency.shape,
)

repeat_lineage_consistency.head()

Repeat-level lineage-consistency rows: (1875, 7)


,DRUG_ID,repeat,program,pooled_mean_abs_shap,n_lineages,lineage_balanced_mean_abs_shap,pooled_minus_lineage_balanced
0,1003,1,CONSENSUS_TX_01,0.495195,11,0.457661,0.037534
1,1003,1,CONSENSUS_TX_02,0.280192,11,0.299741,-0.019549
2,1003,1,CONSENSUS_TX_03,0.098533,11,0.106032,-0.007499
3,1003,2,CONSENSUS_TX_01,0.493774,11,0.456187,0.037587
4,1003,2,CONSENSUS_TX_02,0.280817,11,0.300766,-0.019950


In [18]:
# =============================================================================
# Summarize lineage-consistency metrics across repeats
# =============================================================================

drug_program_lineage_consistency = (
    repeat_lineage_consistency
    .groupby(
        [
            "DRUG_ID",
            "program",
        ],
        as_index=False,
    )
    .agg(
        median_pooled_mean_abs_shap=(
            "pooled_mean_abs_shap",
            "median",
        ),
        median_lineage_balanced_mean_abs_shap=(
            "lineage_balanced_mean_abs_shap",
            "median",
        ),
        median_pooled_minus_lineage_balanced=(
            "pooled_minus_lineage_balanced",
            "median",
        ),
        n_lineages=(
            "n_lineages",
            "median",
        ),
    )
)

print(
    "Drug × program lineage-consistency rows:",
    drug_program_lineage_consistency.shape,
)

drug_program_lineage_consistency.head()

Drug × program lineage-consistency rows: (375, 6)


,DRUG_ID,program,median_pooled_mean_abs_shap,median_lineage_balanced_mean_abs_shap,median_pooled_minus_lineage_balanced,n_lineages
0,1003,CONSENSUS_TX_01,0.493774,0.456782,0.037534,11.0
1,1003,CONSENSUS_TX_02,0.280817,0.300802,-0.020876,11.0
2,1003,CONSENSUS_TX_03,0.098533,0.106032,-0.007669,11.0
3,1004,CONSENSUS_TX_01,0.285828,0.266914,0.020235,10.0
4,1004,CONSENSUS_TX_02,0.706181,0.704393,0.001467,10.0


In [19]:
# =============================================================================
# Add lineage-consistency metrics to the integrated drug-program summary
# =============================================================================

drug_program_summary = (
    drug_program_summary
    .merge(
        drug_program_lineage_consistency,
        on=[
            "DRUG_ID",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
)

print(
    "Integrated drug × program summary shape:",
    drug_program_summary.shape,
)

print(
    "Rows missing lineage-consistency summaries:",
    drug_program_summary[
        "median_lineage_balanced_mean_abs_shap"
    ].isna().sum(),
)

drug_program_summary.head()

Integrated drug × program summary shape:

 (375, 23)
Rows missing lineage-consistency summaries: 0


,DRUG_ID,program,median_repeat_mean_abs_shap,q25_repeat_mean_abs_shap,q75_repeat_mean_abs_shap,min_repeat_mean_abs_shap,max_repeat_mean_abs_shap,median_repeat_median_abs_shap,n_repeats,dominant_repeat_count,...,median_coefficient,q25_coefficient,q75_coefficient,fraction_positive,fraction_negative,iqr_coefficient,median_pooled_mean_abs_shap,median_lineage_balanced_mean_abs_shap,median_pooled_minus_lineage_balanced,n_lineages
0,1003,CONSENSUS_TX_01,0.493774,0.493587,0.495094,0.492472,0.495195,0.354222,5,5,...,-0.568176,-0.653374,-0.496686,0.00,1.00,0.156688,0.493774,0.456782,0.037534,11.0
1,1003,CONSENSUS_TX_02,0.280817,0.280192,0.282048,0.279926,0.285687,0.250155,5,0,...,-0.354836,-0.367466,-0.303957,0.00,1.00,0.063509,0.280817,0.300802,-0.020876,11.0
2,1003,CONSENSUS_TX_03,0.098533,0.096758,0.100502,0.095830,0.102058,0.068122,5,0,...,0.164435,0.091670,0.178533,0.96,0.04,0.086863,0.098533,0.106032,-0.007669,11.0
3,1004,CONSENSUS_TX_01,0.285828,0.283541,0.288258,0.282115,0.288843,0.209169,5,0,...,-0.372261,-0.442923,-0.226922,0.00,1.00,0.216001,0.285828,0.266914,0.020235,10.0
4,1004,CONSENSUS_TX_02,0.706181,0.706094,0.707038,0.702981,0.710803,0.650627,5,5,...,-0.892103,-1.026734,-0.864548,0.00,1.00,0.162186,0.706181,0.704393,0.001467,10.0


In [20]:
# =============================================================================
# Summarize lineage-specific attribution across repeats
# =============================================================================

drug_lineage_program_summary = (
    lineage_program_attribution
    .groupby(
        [
            "DRUG_ID",
            "OncotreeLineage",
            "program",
        ],
        as_index=False,
    )
    .agg(
        median_lineage_mean_abs_shap=(
            "lineage_mean_abs_shap",
            "median",
        ),
        n_repeats=(
            "repeat",
            "nunique",
        ),
    )
)

print(
    "Drug × lineage × program summary rows:",
    drug_lineage_program_summary.shape,
)

drug_lineage_program_summary.head()

Drug × lineage × program summary rows: (3918, 5)


,DRUG_ID,OncotreeLineage,program,median_lineage_mean_abs_shap,n_repeats
0,1003,Bowel,CONSENSUS_TX_01,0.242381,5
1,1003,Bowel,CONSENSUS_TX_02,0.270388,5
2,1003,Bowel,CONSENSUS_TX_03,0.114279,5
3,1003,Breast,CONSENSUS_TX_01,0.320671,5
4,1003,Breast,CONSENSUS_TX_02,0.255379,5


In [21]:
# =============================================================================
# Resolve frozen biological-context handoffs
# =============================================================================

BIOLOGICAL_CONTEXT_ARTIFACT_IDS = (
    "phase4.401.consensus_transcriptomic_program_catalog",
    "phase4.401.consensus_transcriptomic_gene_weights",
    "phase4.401.consensus_tumor_arm_context",
    "phase4.403.epigenetic_regulator_enrichment_summary",
    "phase4.403.epigenetic_regulator_gene_context",
    "phase4.404.program_annotation_enrichment",
    "phase4b.450.primary_gene_program_associations",
    "phase4b.451.locus_level_methylation_expression_evidence",
)

biological_context_paths = {
    artifact_id: resolve_artifact_path(
        artifact_registry,
        artifact_id,
    )
    for artifact_id in BIOLOGICAL_CONTEXT_ARTIFACT_IDS
}

for artifact_id, path in biological_context_paths.items():
    print(
        f"{artifact_id}: "
        f"{project_relative_path(path)}"
    )

phase4.401.consensus_transcriptomic_program_catalog: data/processed/consensus_programs/401_consensus_transcriptomic_program_catalog.csv
phase4.401.consensus_transcriptomic_gene_weights: data/processed/consensus_programs/401_consensus_transcriptomic_gene_weights.csv
phase4.401.consensus_tumor_arm_context: data/processed/consensus_programs/401_consensus_tumor_arm_context.csv
phase4.403.epigenetic_regulator_enrichment_summary: data/processed/consensus_programs/403_epigenetic_regulator_enrichment_summary.csv
phase4.403.epigenetic_regulator_gene_context: data/processed/consensus_programs/403_epigenetic_regulator_gene_context.csv
phase4.404.program_annotation_enrichment: data/processed/consensus_programs/404_program_annotation_enrichment.csv
phase4b.450.primary_gene_program_associations: data/processed/secondary_characterization/450_primary_gene_program_associations.csv
phase4b.451.locus_level_methylation_expression_evidence: data/processed/secondary_characterization/451_locus_level_methylat

In [22]:
# =============================================================================
# Load frozen biological-context sources
# =============================================================================

program_catalog = pd.read_csv(
    biological_context_paths[
        "phase4.401.consensus_transcriptomic_program_catalog"
    ]
)

program_gene_weights = pd.read_csv(
    biological_context_paths[
        "phase4.401.consensus_transcriptomic_gene_weights"
    ]
)

tumor_arm_context = pd.read_csv(
    biological_context_paths[
        "phase4.401.consensus_tumor_arm_context"
    ]
)

epigenetic_enrichment = pd.read_csv(
    biological_context_paths[
        "phase4.403.epigenetic_regulator_enrichment_summary"
    ]
)

epigenetic_gene_context = pd.read_csv(
    biological_context_paths[
        "phase4.403.epigenetic_regulator_gene_context"
    ]
)

program_annotation = pd.read_csv(
    biological_context_paths[
        "phase4.404.program_annotation_enrichment"
    ]
)

gene_program_associations = pd.read_csv(
    biological_context_paths[
        "phase4b.450.primary_gene_program_associations"
    ]
)

locus_methylation_expression = pd.read_csv(
    biological_context_paths[
        "phase4b.451.locus_level_methylation_expression_evidence"
    ]
)

BIOLOGICAL_CONTEXT_TABLES = {
    "program_catalog": program_catalog,
    "program_gene_weights": program_gene_weights,
    "tumor_arm_context": tumor_arm_context,
    "epigenetic_enrichment": epigenetic_enrichment,
    "epigenetic_gene_context": epigenetic_gene_context,
    "program_annotation": program_annotation,
    "gene_program_associations": gene_program_associations,
    "locus_methylation_expression": locus_methylation_expression,
}

for name, table in BIOLOGICAL_CONTEXT_TABLES.items():
    print(f"\n{name} {table.shape}")
    print(table.columns.tolist())


program_catalog (3, 21)
['consensus_program_id', 'tumor_rna_axis', 'cell_line_program', 'orientation_multiplier', 'tumor_arm_count', 'robustness_category', 'program_status', 'context_sensitive', 'association_unstable', 'unresolved_confounding', 'cross_method_convergent', 'tumor_candidate_pairs', 'tumor_methylation_components', 'tumor_structural_families', 'tumor_shared_loading_energy', 'cell_line_shared_loading_energy', 'representation_scope', 'tumor_native_score_pearson', 'tumor_native_score_spearman', 'cell_line_native_score_pearson', 'cell_line_native_score_spearman']

program_gene_weights (7167, 8)
['consensus_program_id', 'gene_symbol', 'tumor_rna_axis', 'cell_line_program', 'orientation_multiplier', 'tumor_loading', 'cell_line_loading', 'consensus_weight']

tumor_arm_context (4, 25)
['consensus_program_id', 'candidate_pair', 'rna_component', 'methylation_component', 'structural_family', 'cell_line_program', 'orientation_multiplier', 'final_audit_status', 'scientific_priority', '

In [23]:
# =============================================================================
# Inspect biological-context identifiers and frozen category fields
# =============================================================================

print(
    "Program catalog IDs:",
    program_catalog["consensus_program_id"].tolist(),
)

print(
    "\nPhase 4B gene-association program IDs:",
    sorted(
        gene_program_associations["program"]
        .dropna()
        .unique()
        .tolist()
    ),
)

print(
    "\nEpigenetic enrichment annotation levels:",
    sorted(
        epigenetic_enrichment["annotation_level"]
        .dropna()
        .unique()
        .tolist()
    ),
)

print(
    "\nEpigenetic enrichment analysis tiers:",
    sorted(
        epigenetic_enrichment["analysis_tier"]
        .dropna()
        .unique()
        .tolist()
    ),
)

print(
    "\nProgram-annotation collections:",
    sorted(
        program_annotation["collection"]
        .dropna()
        .unique()
        .tolist()
    ),
)

print(
    "\nProgram-annotation analysis tiers:",
    sorted(
        program_annotation["analysis_tier"]
        .dropna()
        .unique()
        .tolist()
    ),
)

print(
    "\nLocus-level consensus-program memberships:",
    sorted(
        locus_methylation_expression[
            "consensus_program_memberships"
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )[:20],
)

Program catalog IDs: ['CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03']

Phase 4B gene-association program IDs: ['CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03']

Epigenetic enrichment annotation levels: ['protein_complex', 'regulator_class']

Epigenetic enrichment analysis tiers: ['exploratory_descriptive', 'primary', 'primary_descriptive']

Program-annotation collections: ['GO_BP', 'HALLMARK', 'REACTOME']

Program-annotation analysis tiers: ['exploratory', 'primary', 'secondary']

Locus-level consensus-program memberships: ['CONSENSUS_TX_01;CONSENSUS_TX_03', 'CONSENSUS_TX_02', 'CONSENSUS_TX_02;CONSENSUS_TX_03', 'CONSENSUS_TX_03']


## Frozen biological-context integration

Biological contextualization is assembled once for each frozen consensus
program and is reused identically across all notebook-602 drug-level
attributions.

The integration preserves the inferential hierarchy already defined upstream:

- notebook 401 program metadata and tumor-arm context are retained as frozen
  program-definition context;
- notebook 403 regulator-class results retain their original `analysis_tier`;
  `primary` results are inferential, `primary_descriptive` results remain
  descriptive, and protein-complex results remain exploratory descriptive;
- notebook 404 annotations retain the frozen hierarchy:
  Hallmark = primary, Reactome = secondary, GO Biological Process =
  exploratory. FDR-supported annotations are defined only by the upstream
  frozen `q_value < 0.05` rule;
- notebook 450 mutation context uses the frozen
  `cross_cancer_recurrent` designation as recurrent secondary genomic context;
- notebook 451 locus-level context uses the frozen
  `cross_project_recurrent` designation and preserves all upstream sensitivity
  and promoter-context flags;
- multi-program memberships in notebook 451 are expanded only to link the
  existing evidence to each listed consensus program.

No new biological significance threshold, top-N rule, pathway ranking,
gene ranking, or pharmacogenomic selection criterion is introduced in
notebook 602.

These biological-context layers contextualize program-level model attribution.
They do not create gene-level SHAP evidence, causal mechanisms, validated
targets, or therapeutic claims.

In [24]:
# =============================================================================
# Prepare frozen biological-context evidence layers
# =============================================================================

epigenetic_context = epigenetic_enrichment.copy()

supported_annotation_context = (
    program_annotation.loc[
        program_annotation["q_value"].lt(0.05)
    ]
    .copy()
)

recurrent_mutation_context = (
    gene_program_associations.loc[
        gene_program_associations[
            "cross_cancer_recurrent"
        ].astype(bool)
    ]
    .copy()
)

recurrent_locus_context = (
    locus_methylation_expression.loc[
        locus_methylation_expression[
            "cross_project_recurrent"
        ].astype(bool)
    ]
    .copy()
)

recurrent_locus_context[
    "consensus_program_id"
] = (
    recurrent_locus_context[
        "consensus_program_memberships"
    ]
    .astype(str)
    .str.split(";")
)

recurrent_locus_context = (
    recurrent_locus_context
    .explode("consensus_program_id")
    .reset_index(drop=True)
)

print("Epigenetic-context rows:", epigenetic_context.shape)
print(
    "FDR-supported annotation rows:",
    supported_annotation_context.shape,
)
print(
    "Cross-cancer recurrent mutation-context rows:",
    recurrent_mutation_context.shape,
)
print(
    "Expanded recurrent locus-context rows:",
    recurrent_locus_context.shape,
)

Epigenetic-context rows: (237, 17)
FDR-supported annotation rows: (1114, 10)
Cross-cancer recurrent mutation-context rows: (1601, 30)
Expanded recurrent locus-context rows: (325, 57)


In [25]:
# =============================================================================
# Summarize frozen biological-context layers by consensus program
# =============================================================================

gene_weight_context_summary = (
    program_gene_weights
    .groupby("consensus_program_id", as_index=False)
    .agg(
        n_consensus_genes=("gene_symbol", "nunique"),
    )
)

tumor_arm_context_summary = (
    tumor_arm_context
    .groupby("consensus_program_id", as_index=False)
    .agg(
        n_tumor_arm_context_records=("candidate_pair", "size"),
        n_tumor_candidate_pairs=("candidate_pair", "nunique"),
        n_tumor_methylation_components=(
            "methylation_component",
            "nunique",
        ),
    )
)

epigenetic_context = epigenetic_context.assign(
    primary_fdr_supported=(
        epigenetic_context["analysis_tier"].eq("primary")
        & epigenetic_context["bh_q_value"].lt(0.05)
    )
)

epigenetic_context_summary = (
    epigenetic_context
    .groupby("consensus_program_id", as_index=False)
    .agg(
        n_epigenetic_context_records=(
            "annotation_name",
            "size",
        ),
        n_primary_regulator_tests=(
            "analysis_tier",
            lambda x: x.eq("primary").sum(),
        ),
        n_primary_regulator_fdr_supported=(
            "primary_fdr_supported",
            "sum",
        ),
        n_primary_descriptive_regulator_context=(
            "analysis_tier",
            lambda x: x.eq("primary_descriptive").sum(),
        ),
        n_exploratory_complex_context=(
            "analysis_tier",
            lambda x: x.eq("exploratory_descriptive").sum(),
        ),
    )
)

epigenetic_gene_context_summary = (
    epigenetic_gene_context
    .groupby("consensus_program_id", as_index=False)
    .agg(
        n_epigenetic_regulator_genes=(
            "gene_symbol",
            "nunique",
        ),
    )
)

annotation_context_summary = (
    supported_annotation_context
    .groupby(
        [
            "consensus_program_id",
            "collection",
        ]
    )
    .size()
    .unstack(fill_value=0)
    .rename(
        columns={
            "HALLMARK": "n_supported_hallmark",
            "REACTOME": "n_supported_reactome",
            "GO_BP": "n_supported_go_bp",
        }
    )
    .reset_index()
)

mutation_context_summary = (
    recurrent_mutation_context
    .groupby("program", as_index=False)
    .agg(
        n_recurrent_mutation_pairs=(
            "Hugo_Symbol",
            "size",
        ),
        n_unique_recurrent_mutation_genes=(
            "Hugo_Symbol",
            "nunique",
        ),
    )
    .rename(
        columns={
            "program": "consensus_program_id",
        }
    )
)

locus_context_summary = (
    recurrent_locus_context
    .groupby("consensus_program_id", as_index=False)
    .agg(
        n_recurrent_locus_memberships=(
            "probe_id",
            "size",
        ),
        n_unique_recurrent_cpgs=(
            "probe_id",
            "nunique",
        ),
        n_unique_recurrent_expression_genes=(
            "rna_gene_id_base",
            "nunique",
        ),
    )
)

for name, table in {
    "gene weights": gene_weight_context_summary,
    "tumor arm": tumor_arm_context_summary,
    "epigenetic enrichment": epigenetic_context_summary,
    "epigenetic genes": epigenetic_gene_context_summary,
    "annotations": annotation_context_summary,
    "mutation context": mutation_context_summary,
    "locus context": locus_context_summary,
}.items():
    print(f"{name}: {table.shape}")

gene weights: (3, 2)
tumor arm: (3, 4)
epigenetic enrichment: (3, 6)
epigenetic genes: (3, 2)
annotations: (3, 4)
mutation context: (3, 3)
locus context: (3, 4)


In [26]:
# =============================================================================
# Assemble program-level biological context
# =============================================================================

program_biological_context = (
    program_catalog[
        [
            "consensus_program_id",
            "tumor_rna_axis",
            "cell_line_program",
            "orientation_multiplier",
            "tumor_arm_count",
            "robustness_category",
            "program_status",
            "context_sensitive",
            "association_unstable",
            "unresolved_confounding",
            "cross_method_convergent",
            "representation_scope",
        ]
    ]
    .merge(
        gene_weight_context_summary,
        on="consensus_program_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        tumor_arm_context_summary,
        on="consensus_program_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        epigenetic_context_summary,
        on="consensus_program_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        epigenetic_gene_context_summary,
        on="consensus_program_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        annotation_context_summary,
        on="consensus_program_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        mutation_context_summary,
        on="consensus_program_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        locus_context_summary,
        on="consensus_program_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("consensus_program_id")
    .reset_index(drop=True)
)

print(
    "Program biological-context shape:",
    program_biological_context.shape,
)

program_biological_context

Program biological-context shape: (3, 30)


,consensus_program_id,tumor_rna_axis,cell_line_program,orientation_multiplier,tumor_arm_count,robustness_category,program_status,context_sensitive,association_unstable,unresolved_confounding,...,n_exploratory_complex_context,n_epigenetic_regulator_genes,n_supported_go_bp,n_supported_hallmark,n_supported_reactome,n_recurrent_mutation_pairs,n_unique_recurrent_mutation_genes,n_recurrent_locus_memberships,n_unique_recurrent_cpgs,n_unique_recurrent_expression_genes
0,CONSENSUS_TX_01,RNA_IC150,ICA_PROGRAM_09,1,1,CONTEXT_SENSITIVE_CANDIDATE,candidate_with_cross_method_support,True,False,False,...,73,32,465,18,78,1315,1315,93,86,90
1,CONSENSUS_TX_02,RNA_IC151,ICA_PROGRAM_29,-1,1,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_with_cross_method_support,False,False,False,...,73,32,34,9,19,78,78,78,73,61
2,CONSENSUS_TX_03,RNA_IC184,ICA_PROGRAM_13,-1,2,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_ica_specific,False,False,False,...,73,32,426,12,53,208,208,154,142,142


In [ ]:
# =============================================================================
# Characterize global attribution patterns across eligible drugs
# =============================================================================

program_level_results = (
    drug_program_summary
    .assign(
        abs_pooled_minus_lineage_balanced=lambda df: (
            df["median_pooled_minus_lineage_balanced"].abs()
        )
    )
    .groupby("program", as_index=False)
    .agg(
        n_drugs=("DRUG_ID", "nunique"),
        median_attribution=(
            "median_repeat_mean_abs_shap",
            "median",
        ),
        q25_attribution=(
            "median_repeat_mean_abs_shap",
            lambda x: x.quantile(0.25),
        ),
        q75_attribution=(
            "median_repeat_mean_abs_shap",
            lambda x: x.quantile(0.75),
        ),
        median_repeat_iqr=(
            "iqr_repeat_mean_abs_shap",
            "median",
        ),
        drugs_dominant_all_repeats=(
            "dominant_repeat_fraction",
            lambda x: x.eq(1.0).sum(),
        ),
        total_dominant_repeats=(
            "dominant_repeat_count",
            "sum",
        ),
        median_coefficient=(
            "median_coefficient",
            "median",
        ),
        median_fraction_positive=(
            "fraction_positive",
            "median",
        ),
        median_fraction_negative=(
            "fraction_negative",
            "median",
        ),
        median_lineage_composition_difference=(
            "abs_pooled_minus_lineage_balanced",
            "median",
        ),
    )
)

program_level_results[
    "attribution_iqr"
] = (
    program_level_results["q75_attribution"]
    - program_level_results["q25_attribution"]
)

program_level_results

In [ ]:
# =============================================================================
# Compare program-block and lineage-block attribution magnitude
# =============================================================================

attribution_rows["phi_program_block"] = (
    attribution_rows[
        [
            "phi_CONSENSUS_TX_01",
            "phi_CONSENSUS_TX_02",
            "phi_CONSENSUS_TX_03",
        ]
    ]
    .sum(axis=1)
)

repeat_block_attribution = (
    attribution_rows
    .assign(
        abs_phi_program_block=lambda df: (
            df["phi_program_block"].abs()
        ),
        abs_phi_lineage=lambda df: (
            df["phi_lineage"].abs()
        ),
    )
    .groupby(
        [
            "DRUG_ID",
            "repeat",
        ],
        as_index=False,
    )
    .agg(
        mean_abs_program_block=(
            "abs_phi_program_block",
            "mean",
        ),
        mean_abs_lineage=(
            "abs_phi_lineage",
            "mean",
        ),
    )
)

drug_block_attribution = (
    repeat_block_attribution
    .groupby(
        "DRUG_ID",
        as_index=False,
    )
    .agg(
        median_mean_abs_program_block=(
            "mean_abs_program_block",
            "median",
        ),
        median_mean_abs_lineage=(
            "mean_abs_lineage",
            "median",
        ),
    )
)

drug_block_attribution[
    "program_minus_lineage_attribution"
] = (
    drug_block_attribution[
        "median_mean_abs_program_block"
    ]
    - drug_block_attribution[
        "median_mean_abs_lineage"
    ]
)

print(
    "Drug-level block-attribution rows:",
    drug_block_attribution.shape,
)

display(
    drug_block_attribution[
        [
            "median_mean_abs_program_block",
            "median_mean_abs_lineage",
            "program_minus_lineage_attribution",
        ]
    ]
    .describe()
)

In [ ]:
# =============================================================================
# Characterize program-block versus lineage-block attribution across repeats
# =============================================================================

repeat_block_attribution[
    "program_block_exceeds_lineage"
] = (
    repeat_block_attribution[
        "mean_abs_program_block"
    ]
    > repeat_block_attribution[
        "mean_abs_lineage"
    ]
)

drug_block_consistency = (
    repeat_block_attribution
    .groupby(
        "DRUG_ID",
        as_index=False,
    )
    .agg(
        repeats_program_block_exceeds_lineage=(
            "program_block_exceeds_lineage",
            "sum",
        ),
    )
)

drug_block_consistency[
    "fraction_repeats_program_block_exceeds_lineage"
] = (
    drug_block_consistency[
        "repeats_program_block_exceeds_lineage"
    ]
    / 5
)

print(
    "Drugs with program block > lineage in all 5 repeats:",
    (
        drug_block_consistency[
            "repeats_program_block_exceeds_lineage"
        ]
        .eq(5)
        .sum()
    ),
)

print(
    "Drugs with program block > lineage in at least 3/5 repeats:",
    (
        drug_block_consistency[
            "repeats_program_block_exceeds_lineage"
        ]
        .ge(3)
        .sum()
    ),
)

print(
    "Drugs with lineage >= program block in all 5 repeats:",
    (
        drug_block_consistency[
            "repeats_program_block_exceeds_lineage"
        ]
        .eq(0)
        .sum()
    ),
)

print(
    "\nDistribution of repeat consistency:"
)
print(
    drug_block_consistency[
        "repeats_program_block_exceeds_lineage"
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# =============================================================================
# Characterize coefficient-direction stability across eligible drugs
# =============================================================================

coefficient_direction_results = (
    drug_program_summary
    .assign(
        all_positive=lambda df: (
            df["fraction_positive"].eq(1.0)
        ),
        all_negative=lambda df: (
            df["fraction_negative"].eq(1.0)
        ),
        direction_mixed=lambda df: (
            ~df["fraction_positive"].eq(1.0)
            & ~df["fraction_negative"].eq(1.0)
        ),
    )
    .groupby("program", as_index=False)
    .agg(
        n_drugs=("DRUG_ID", "nunique"),
        all_positive_drugs=("all_positive", "sum"),
        all_negative_drugs=("all_negative", "sum"),
        mixed_direction_drugs=("direction_mixed", "sum"),
        median_fraction_positive=(
            "fraction_positive",
            "median",
        ),
        median_fraction_negative=(
            "fraction_negative",
            "median",
        ),
        median_coefficient=(
            "median_coefficient",
            "median",
        ),
        median_coefficient_iqr=(
            "iqr_coefficient",
            "median",
        ),
    )
)

coefficient_direction_results

In [ ]:
# =============================================================================
# Characterize lineage-composition sensitivity across eligible drugs
# =============================================================================

lineage_composition_results = (
    drug_program_lineage_consistency
    .assign(
        pooled_higher=lambda df: (
            df["median_pooled_minus_lineage_balanced"] > 0
        ),
        balanced_higher=lambda df: (
            df["median_pooled_minus_lineage_balanced"] < 0
        ),
    )
    .groupby("program", as_index=False)
    .agg(
        n_drugs=("DRUG_ID", "nunique"),
        median_difference=(
            "median_pooled_minus_lineage_balanced",
            "median",
        ),
        q25_difference=(
            "median_pooled_minus_lineage_balanced",
            lambda x: x.quantile(0.25),
        ),
        q75_difference=(
            "median_pooled_minus_lineage_balanced",
            lambda x: x.quantile(0.75),
        ),
        pooled_higher_drugs=(
            "pooled_higher",
            "sum",
        ),
        balanced_higher_drugs=(
            "balanced_higher",
            "sum",
        ),
    )
)

lineage_composition_results

In [ ]:
# =============================================================================
# Evaluate whether lineage balancing changes dominant program identity
# =============================================================================

dominance_input = (
    drug_program_lineage_consistency[
        [
            "DRUG_ID",
            "program",
            "median_pooled_mean_abs_shap",
            "median_lineage_balanced_mean_abs_shap",
        ]
    ]
    .copy()
)

dominance_input["pooled_dominant"] = (
    dominance_input["median_pooled_mean_abs_shap"]
    == dominance_input.groupby("DRUG_ID")[
        "median_pooled_mean_abs_shap"
    ].transform("max")
)

dominance_input["lineage_balanced_dominant"] = (
    dominance_input["median_lineage_balanced_mean_abs_shap"]
    == dominance_input.groupby("DRUG_ID")[
        "median_lineage_balanced_mean_abs_shap"
    ].transform("max")
)

pooled_ties = (
    dominance_input.loc[
        dominance_input["pooled_dominant"]
    ]
    .groupby("DRUG_ID")
    .size()
    .gt(1)
    .sum()
)

balanced_ties = (
    dominance_input.loc[
        dominance_input["lineage_balanced_dominant"]
    ]
    .groupby("DRUG_ID")
    .size()
    .gt(1)
    .sum()
)

pooled_dominant = (
    dominance_input.loc[
        dominance_input["pooled_dominant"],
        ["DRUG_ID", "program"],
    ]
    .rename(
        columns={"program": "pooled_dominant_program"}
    )
)

balanced_dominant = (
    dominance_input.loc[
        dominance_input["lineage_balanced_dominant"],
        ["DRUG_ID", "program"],
    ]
    .rename(
        columns={
            "program": "lineage_balanced_dominant_program"
        }
    )
)

dominance_comparison = pooled_dominant.merge(
    balanced_dominant,
    on="DRUG_ID",
    how="inner",
    validate="one_to_one",
)

dominance_comparison[
    "dominant_program_changed"
] = (
    dominance_comparison[
        "pooled_dominant_program"
    ]
    != dominance_comparison[
        "lineage_balanced_dominant_program"
    ]
)

balanced_program_summary = (
    drug_program_lineage_consistency
    .groupby("program", as_index=False)
    .agg(
        median_lineage_balanced_attribution=(
            "median_lineage_balanced_mean_abs_shap",
            "median",
        ),
    )
)

print("Pooled-dominance ties:", pooled_ties)
print("Lineage-balanced dominance ties:", balanced_ties)

print(
    "\nDrugs changing dominant program after lineage balancing:",
    dominance_comparison[
        "dominant_program_changed"
    ].sum(),
)

print("\nPooled dominant programs:")
print(
    dominance_comparison[
        "pooled_dominant_program"
    ]
    .value_counts()
)

print("\nLineage-balanced dominant programs:")
print(
    dominance_comparison[
        "lineage_balanced_dominant_program"
    ]
    .value_counts()
)

print("\nDominance transitions:")
print(
    pd.crosstab(
        dominance_comparison[
            "pooled_dominant_program"
        ],
        dominance_comparison[
            "lineage_balanced_dominant_program"
        ],
    )
)

print("\nLineage-balanced attribution across drugs:")
print(balanced_program_summary)

## Results and scientific interpretation

Notebook 602 characterized program-level attribution for the 125 GDSC drugs that satisfied the prospectively frozen notebook-601 predictive-validity gate. Attribution was calculated exclusively from persisted held-out predictions and fitted-state parameters; no model was refitted and no additional eligibility criterion was introduced.

### Program-level attribution magnitude

The three frozen consensus transcriptomic programs contributed unequally to held-out model predictions.

Across the 125 eligible drugs, the median drug-level attribution magnitude (`median_repeat_mean_abs_shap`) was:

* `CONSENSUS_TX_01`: 0.200
* `CONSENSUS_TX_02`: 0.281
* `CONSENSUS_TX_03`: 0.140

`CONSENSUS_TX_02` therefore showed the highest median attribution magnitude in this internally selected GDSC predictive cohort.

Program dominance was also strongly structured across compounds. The same program had the largest mean absolute attribution in all five repeats for 123/125 drugs:

* `CONSENSUS_TX_02`: 78/125 drugs
* `CONSENSUS_TX_01`: 38/125 drugs
* `CONSENSUS_TX_03`: 7/125 drugs

Only two drugs did not retain a single dominant program across all five repeats.

Repeat-level attribution magnitude was numerically stable. The median interquartile range of repeat-level `mean_abs_shap` values was approximately 0.0021 for `CONSENSUS_TX_01`, 0.0014 for `CONSENSUS_TX_02`, and 0.0021 for `CONSENSUS_TX_03`. This indicates limited sensitivity of aggregate attribution magnitude to the repeated cross-validation partitioning used in notebook 601.

These results describe fitted-model attribution within the 125-drug cohort that already satisfied the notebook-601 predictive-validity criteria. They do not establish that a highly attributed program is biologically causal, therapeutically relevant, or externally reproducible.

### Direction of fitted program relationships

Direction was evaluated from the 25 persisted fold-specific coefficients per drug and program rather than from signed SHAP values.

`CONSENSUS_TX_02` showed the strongest within-drug directional stability. All 125 drugs retained a single coefficient direction across all 25 fitted models: 112 were consistently negative and 13 consistently positive.

For `CONSENSUS_TX_01`, 88 drugs were consistently negative, 7 consistently positive, and 30 showed coefficient sign variation across fitted models.

For `CONSENSUS_TX_03`, 95 drugs were consistently positive, 7 consistently negative, and 23 showed mixed coefficient direction.

Across drugs, the median fitted coefficient was negative for `CONSENSUS_TX_01` (-0.209) and `CONSENSUS_TX_02` (-0.307), and positive for `CONSENSUS_TX_03` (+0.185).

Under the frozen response convention, a positive coefficient indicates that higher program score is associated with higher predicted `LN_IC50` after accounting for lineage and the other two programs, whereas a negative coefficient indicates association with lower predicted `LN_IC50`. These fitted-model relationships must not be interpreted as causal effects of experimentally increasing or decreasing the biological programs.

### Program-block versus lineage attribution

The combined transcriptomic-program contribution remained substantial relative to the lineage component among the 125 drugs selected for attribution.

Across drugs, the median of the repeat-level mean absolute program-block attribution was 0.411 `LN_IC50` units, compared with 0.345 for the grouped lineage attribution. The median difference was +0.059, although the drug-level range was broad (-0.501 to +0.488).

For 75/125 drugs, the program block had larger mean absolute attribution than lineage in all five repeats. For 46/125 drugs, lineage was at least as large as the program block in all five repeats. Only four drugs showed an intermediate repeat-dependent pattern.

This comparison is a prediction-space attribution characterization. It is not an `R²` decomposition and does not replace the notebook-601 finding that the median incremental predictive contribution of the three programs beyond lineage was modest (`median ΔR² = 0.016`) across the complete 281-drug predictive cohort.

Together, notebooks 601 and 602 indicate that lineage remains an important source of pharmacogenomic predictive structure, while the frozen transcriptomic programs contribute additional prediction-space information for the prospectively selected 125-drug attribution cohort.

### Lineage-composition sensitivity

Lineage-balanced attribution produced moderate shifts in the global program summaries, with systematic program-specific direction.

For `CONSENSUS_TX_01`, pooled attribution exceeded the equal-lineage-weighted summary for all 125 drugs, with a median pooled-minus-lineage-balanced difference of +0.0147.

For `CONSENSUS_TX_02`, the lineage-balanced value exceeded the pooled value for 113/125 drugs, with a median difference of -0.0110.

For `CONSENSUS_TX_03`, the lineage-balanced value exceeded the pooled value for 124/125 drugs, with a median difference of -0.0113.

Thus, observed cell-line composition affects estimated attribution magnitude in a program-dependent manner. Equal weighting of represented lineages tends to reduce the estimated attribution of `CONSENSUS_TX_01` and increase that of `CONSENSUS_TX_02` and `CONSENSUS_TX_03`.

These differences do not constitute program × lineage interactions. The fitted model contains no such interaction terms, and lineage-stratified SHAP summaries characterize the distribution of model attribution across represented lineages rather than lineage-specific program effects.

### Lineage-balanced dominance sensitivity

The identity of the highest-attribution program was largely insensitive to unequal lineage representation within individual drug cohorts.

Using the pooled attribution summaries, `CONSENSUS_TX_02` was the highest-attribution program for 80/125 drugs, `CONSENSUS_TX_01` for 38/125, and `CONSENSUS_TX_03` for 7/125.

After assigning equal weight to each represented lineage, only 4/125 drugs changed dominant-program identity. The lineage-balanced counts were 82/125 for `CONSENSUS_TX_02`, 35/125 for `CONSENSUS_TX_01`, and 8/125 for `CONSENSUS_TX_03`.

Median lineage-balanced attribution across drugs remained highest for `CONSENSUS_TX_02` (0.292), followed by `CONSENSUS_TX_01` (0.182) and `CONSENSUS_TX_03` (0.150).

Thus, although lineage composition produces systematic program-specific shifts in attribution magnitude, unequal representation of lineages does not materially alter the overall dominant-program structure within this internally selected GDSC cohort.

This sensitivity analysis addresses weighting by represented lineage size only. It does not establish lineage independence, remove residual lineage-related confounding, or estimate program × lineage interactions.

### Biological context

The attribution results retain the biological identities established in the frozen Phase 4 analyses.

`CONSENSUS_TX_01` corresponds to an immune-associated transcriptomic axis with upstream context sensitivity. `CONSENSUS_TX_02` has a more restricted and asymmetric annotation structure, including negative representation of keratinization/epidermal-differentiation and interferon-related processes. `CONSENSUS_TX_03` represents a strong extracellular-matrix and mesenchymal-associated axis.

Notebook 602 additionally links each program to the frozen epigenetic-regulator, tumor-side methylation, pathway-annotation, recurrent somatic-mutation, and locus-level methylation-expression context generated in Phases 4 and 4B.

These layers contextualize program-level attribution but do not convert SHAP evidence into gene-level attribution, biological mechanism, validated targets, or therapeutic evidence.

### Overall interpretation

Notebook 602 identifies a structured and resampling-stable pattern of model attribution among the 125 GDSC drugs that passed the frozen internal predictive-validity gate.

`CONSENSUS_TX_02` shows the largest aggregate attribution and the strongest coefficient-direction stability across this cohort. `CONSENSUS_TX_01` contributes intermediate attribution with greater directional heterogeneity, while `CONSENSUS_TX_03` has lower aggregate attribution and predominantly positive fitted coefficients.

The combined transcriptomic-program block exceeds the lineage attribution component consistently for a majority of eligible drugs, but lineage remains dominant for a substantial subset. Attribution magnitude also shows systematic but moderate sensitivity to the composition of represented cancer lineages, while dominant-program identity is largely preserved after equal-lineage weighting.

These findings support interpretable, resampling-stable program-level contributions to internally valid GDSC pharmacogenomic models. They remain developmental evidence within GDSC. External cross-screen reproducibility in CTRP and PRISM has not yet been evaluated and is reserved for notebook 603.

No result in notebook 602 establishes causal drug-response mechanisms, therapeutic efficacy, clinical resistance prediction, validated biomarkers, or validated therapeutic targets.

## Stable outputs and closeout

The analytical interpretation above is completed before persistence so that all stable notebook-602 handoffs reflect the final executed analysis.

The stable tabular interfaces include held-out program attributions, repeat-level summaries, lineage-level summaries, drug × program summaries, the supporting drug-level program-block versus lineage comparison, and the compact program-level biological-context index.

Persistence and metadata below do not introduce new analytical decisions or reinterpret upstream evidence.

In [ ]:
# =============================================================================
# Prepare stable notebook-602 output tables
# =============================================================================

oof_program_attributions_output = (
    attribution_rows[
        [
            "DRUG_ID",
            "ModelID",
            "OncotreeLineage",
            "repeat",
            "fold",
            "prediction_program",
            "expected_value",
            "phi_lineage",
            "phi_CONSENSUS_TX_01",
            "phi_CONSENSUS_TX_02",
            "phi_CONSENSUS_TX_03",
        ]
    ]
    .sort_values(
        [
            "DRUG_ID",
            "repeat",
            "fold",
            "ModelID",
        ]
    )
    .reset_index(drop=True)
)

repeat_program_attribution_output = (
    repeat_program_attribution
    .merge(
        repeat_lineage_consistency[
            [
                "DRUG_ID",
                "repeat",
                "program",
                "lineage_balanced_mean_abs_shap",
                "pooled_minus_lineage_balanced",
                "n_lineages",
            ]
        ],
        on=[
            "DRUG_ID",
            "repeat",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "DRUG_ID",
            "repeat",
            "program",
        ]
    )
    .reset_index(drop=True)
)

lineage_program_attribution_output = (
    drug_lineage_program_summary
    .sort_values(
        [
            "DRUG_ID",
            "OncotreeLineage",
            "program",
        ]
    )
    .reset_index(drop=True)
)

drug_program_attribution_summary_output = (
    drug_program_summary
    .sort_values(
        [
            "DRUG_ID",
            "program",
        ]
    )
    .reset_index(drop=True)
)

drug_block_attribution_summary_output = (
    drug_block_attribution
    .merge(
        drug_block_consistency,
        on="DRUG_ID",
        how="left",
        validate="one_to_one",
    )
    .sort_values("DRUG_ID")
    .reset_index(drop=True)
)

program_biological_context_output = (
    program_biological_context
    .sort_values("consensus_program_id")
    .reset_index(drop=True)
)

print(
    "OOF program attributions:",
    oof_program_attributions_output.shape,
)
print(
    "Repeat program attribution:",
    repeat_program_attribution_output.shape,
)
print(
    "Lineage program attribution:",
    lineage_program_attribution_output.shape,
)
print(
    "Drug-program attribution summary:",
    drug_program_attribution_summary_output.shape,
)
print(
    "Drug block-attribution summary:",
    drug_block_attribution_summary_output.shape,
)
print(
    "Program biological context:",
    program_biological_context_output.shape,
)

In [ ]:
# =============================================================================
# Attach frozen source identities to program-level biological context
# =============================================================================

program_biological_context_output = (
    program_biological_context_output
    .assign(
        program_definition_source=(
            "phase4.401.consensus_transcriptomic_program_catalog"
        ),
        gene_weight_source=(
            "phase4.401.consensus_transcriptomic_gene_weights"
        ),
        tumor_arm_source=(
            "phase4.401.consensus_tumor_arm_context"
        ),
        epigenetic_enrichment_source=(
            "phase4.403.epigenetic_regulator_enrichment_summary"
        ),
        epigenetic_gene_source=(
            "phase4.403.epigenetic_regulator_gene_context"
        ),
        annotation_source=(
            "phase4.404.program_annotation_enrichment"
        ),
        mutation_context_source=(
            "phase4b.450.primary_gene_program_associations"
        ),
        locus_context_source=(
            "phase4b.451.locus_level_methylation_expression_evidence"
        ),
    )
)

print(
    "Program biological-context shape:",
    program_biological_context_output.shape,
)

In [ ]:
# =============================================================================
# Persist stable notebook-602 tabular outputs
# =============================================================================

output_dir = Paths.pharmacogenomic_contexts
output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

output_paths = {
    "oof_program_attributions": (
        output_dir
        / "602_oof_program_attributions.parquet"
    ),
    "repeat_program_attribution": (
        output_dir
        / "602_repeat_program_attribution.csv"
    ),
    "lineage_program_attribution": (
        output_dir
        / "602_lineage_program_attribution.csv"
    ),
    "drug_program_attribution_summary": (
        output_dir
        / "602_drug_program_attribution_summary.csv"
    ),
    "drug_block_attribution_summary": (
        output_dir
        / "602_drug_block_attribution_summary.csv"
    ),
    "program_biological_context": (
        output_dir
        / "602_program_biological_context.csv"
    ),
}

oof_program_attributions_output.to_parquet(
    output_paths["oof_program_attributions"],
    index=False,
)

repeat_program_attribution_output.to_csv(
    output_paths["repeat_program_attribution"],
    index=False,
)

lineage_program_attribution_output.to_csv(
    output_paths["lineage_program_attribution"],
    index=False,
)

drug_program_attribution_summary_output.to_csv(
    output_paths["drug_program_attribution_summary"],
    index=False,
)

drug_block_attribution_summary_output.to_csv(
    output_paths["drug_block_attribution_summary"],
    index=False,
)

program_biological_context_output.to_csv(
    output_paths["program_biological_context"],
    index=False,
)

for name, path in output_paths.items():
    print(
        f"{name}: "
        f"{project_relative_path(path)}"
    )

In [ ]:
# =============================================================================
# Persist notebook-602 analysis metadata
# =============================================================================

program_summary_records = {
    row.program: {
        "median_attribution": float(row.median_attribution),
        "attribution_iqr": float(row.attribution_iqr),
        "drugs_dominant_all_repeats": int(
            row.drugs_dominant_all_repeats
        ),
        "median_coefficient": float(row.median_coefficient),
    }
    for row in program_level_results.itertuples(index=False)
}

coefficient_direction_records = {
    row.program: {
        "all_positive_drugs": int(row.all_positive_drugs),
        "all_negative_drugs": int(row.all_negative_drugs),
        "mixed_direction_drugs": int(row.mixed_direction_drugs),
    }
    for row in coefficient_direction_results.itertuples(index=False)
}

analysis_metadata = {
    "schema_version": 1,
    "notebook": "602_shap_attribution_and_stability_analysis",
    "analysis_role": "internal_model_attribution",
    "resource": "GDSC",
    "primary_cohort": {
        "source_artifact": "phase6.601.drug_level_results",
        "eligibility_field": "shap_eligible",
        "eligible_drugs": int(len(shap_eligible_drugs)),
        "excluded_drugs": int(
            drug_level_results["DRUG_ID"].nunique()
            - len(shap_eligible_drugs)
        ),
    },
    "attributed_model": (
        "LN_IC50 ~ C(OncotreeLineage) "
        "+ CONSENSUS_TX_01 "
        "+ CONSENSUS_TX_02 "
        "+ CONSENSUS_TX_03"
    ),
    "attribution_definition": {
        "method": "exact_interventional_linear_shap",
        "program_formula": (
            "phi_p = beta_p * (x_p - training_fold_mean_p)"
        ),
        "lineage_formula": (
            "phi_lineage = lineage_effect "
            "- training_mean_lineage_effect"
        ),
        "expected_value_formula": (
            "intercept + training_mean_lineage_effect "
            "+ sum(beta_p * training_fold_mean_p)"
        ),
        "attribution_sample": "persisted_notebook601_oof_predictions",
        "training_attribution_used": False,
        "full_dataset_refit_used": False,
        "program_resolution": PROGRAM_COLUMNS,
        "lineage_component": "grouped_contextual_component",
    },
    "reconstruction_check": {
        "all_predictions_reconstructed": bool(
            reconstruction_matches
        ),
        "maximum_absolute_error": float(
            np.abs(prediction_error).max()
        ),
    },
    "aggregation": {
        "primary_repeat_metric": "mean_abs_shap",
        "primary_drug_program_metric": (
            "median_repeat_mean_abs_shap"
        ),
        "supporting_repeat_metric": "median_abs_shap",
        "repeat_count": 5,
        "binary_shap_stability_gate": False,
        "composite_attribution_score": False,
    },
    "coefficient_direction": {
        "source": (
            "phase6.601.primary_program_fold_parameters"
        ),
        "fitted_models_per_drug_program": 25,
        "summaries": [
            "median_coefficient",
            "iqr_coefficient",
            "fraction_positive",
            "fraction_negative",
        ],
        "independent_replicates": False,
    },
    "lineage_consistency": {
        "lineage_specific_metric": (
            "lineage_mean_abs_shap"
        ),
        "pooled_metric": "pooled_mean_abs_shap",
        "lineage_balanced_metric": (
            "unweighted_mean_of_lineage_mean_abs_shap"
        ),
        "composition_diagnostic": (
            "pooled_minus_lineage_balanced"
        ),
        "dominance_sensitivity": (
            "pooled_vs_lineage_balanced_dominant_program"
        ),
        "new_lineage_specific_models_fitted": False,
        "program_lineage_interactions_fitted": False,
    },
    "program_block_attribution": {
        "definition": (
            "phi_program_block = phi_CONSENSUS_TX_01 "
            "+ phi_CONSENSUS_TX_02 + phi_CONSENSUS_TX_03"
        ),
        "repeat_metric": (
            "mean(abs(phi_program_block)) versus "
            "mean(abs(phi_lineage))"
        ),
        "drug_summary": "median_across_five_repeats",
        "r2_decomposition": False,
        "selection_role": "none",
    },
    "biological_context": {
        "resolution": "consensus_program",
        "sources": list(
            BIOLOGICAL_CONTEXT_ARTIFACT_IDS
        ),
        "gene_level_shap_created": False,
        "program_shap_redistributed_to_genes": False,
        "new_biological_selection_threshold": False,
        "context_reused_across_drugs": True,
    },
    "upstream_artifacts": {
        artifact_id: {
            "path": project_relative_path(
                input_paths[artifact_id]
            ),
            "sha256": artifact_registry[
                "artifacts"
            ][artifact_id]["sha256"],
        }
        for artifact_id in INPUT_ARTIFACT_IDS
    },
    "biological_context_artifacts": {
        artifact_id: {
            "path": project_relative_path(
                biological_context_paths[artifact_id]
            ),
            "sha256": artifact_registry[
                "artifacts"
            ][artifact_id]["sha256"],
        }
        for artifact_id in BIOLOGICAL_CONTEXT_ARTIFACT_IDS
    },
    "execution_summary": {
        "eligible_drugs": int(len(shap_eligible_drugs)),
        "oof_attribution_rows": int(
            len(oof_program_attributions_output)
        ),
        "repeat_program_rows": int(
            len(repeat_program_attribution_output)
        ),
        "lineage_program_rows": int(
            len(lineage_program_attribution_output)
        ),
        "drug_program_summary_rows": int(
            len(drug_program_attribution_summary_output)
        ),
        "drug_block_summary_rows": int(
            len(drug_block_attribution_summary_output)
        ),
        "program_biological_context_rows": int(
            len(program_biological_context_output)
        ),
        "program_block_exceeds_lineage_all_repeats": int(
            drug_block_consistency[
                "repeats_program_block_exceeds_lineage"
            ].eq(5).sum()
        ),
        "lineage_at_least_program_block_all_repeats": int(
            drug_block_consistency[
                "repeats_program_block_exceeds_lineage"
            ].eq(0).sum()
        ),
        "dominant_program_changed_after_lineage_balancing": int(
            dominance_comparison[
                "dominant_program_changed"
            ].sum()
        ),
        "program_summary": program_summary_records,
        "coefficient_direction_summary": (
            coefficient_direction_records
        ),
    },
    "evidence_isolation": {
        "ctrp_outcomes_inspected": False,
        "prism_outcomes_inspected": False,
        "notebook600_associations_used_for_selection": False,
        "phase5_used_for_selection": False,
        "lolo_used_for_attribution_selection": False,
    },
    "interpretation_limitations": [
        "SHAP characterizes fitted-model behavior, not causality.",
        (
            "Interventional attribution terminology does not "
            "imply biological intervention."
        ),
        (
            "Lineage-stratified attribution does not estimate "
            "program-by-lineage interactions."
        ),
        (
            "Repeated folds and repeats are overlapping resampling "
            "structures, not independent biological replicates."
        ),
        (
            "Program-block versus lineage attribution is not an "
            "R2 decomposition."
        ),
        (
            "Biological contextualization does not create "
            "gene-level attribution evidence."
        ),
        (
            "Notebook 602 provides internal GDSC attribution "
            "evidence, not external cross-screen reproducibility."
        ),
    ],
    "outputs": {
        name: project_relative_path(path)
        for name, path in output_paths.items()
    },
}

metadata_path = (
    output_dir
    / "602_analysis_metadata.json"
)

metadata_path.write_text(
    json.dumps(
        analysis_metadata,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print(
    "Analysis metadata:",
    project_relative_path(metadata_path),
)

print(
    "Recorded eligible drugs:",
    analysis_metadata[
        "execution_summary"
    ]["eligible_drugs"],
)

print(
    "Recorded reconstruction maximum absolute error:",
    analysis_metadata[
        "reconstruction_check"
    ]["maximum_absolute_error"],
)